# 08 · 1P1Q: two controls

Notebook 07: the best 1P1Q kernel reaches AUC 0.66, below the encodings of the 3 QGL inputs (0.80). Two questions before interpreting this:

1. **Is it the encoding or the inputs?** Classical models (RBF-SVM, BDT) trained on exactly the inputs that 1P1Q sees: $(z_i, \Delta\eta_i, \Delta\phi_i)$ of the 10 hardest constituents, same subsamples and test set.
2. **Orientation.** Quark/gluon information is in how far constituents are from the jet axis, not in which direction. 1P1Q encodes $\Delta\eta$ and $\Delta\phi$ separately, so two identical jets rotated around the axis map to different states.
   **Rotation-invariant variant (ours, not from the paper):** $\theta_i = f\, z_i\, \Delta R_i$ with $\Delta R_i = \sqrt{\Delta\eta_i^2 + \Delta\phi_i^2}$, one $R_Y(\theta_i)$ per constituent.

In [ ]:
import sys
sys.path.insert(0, "src")

import json
from pathlib import Path
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from jet_data import (load_jets, select_quark_gluon, event_level_split, balanced_subsample,
                      leading_constituents, BOOKKEEPING, PF_KEYS)
from kernels import angle_kernel, shot_noise, concentration, effective_rank

SEED = 42
SEEDS = [0, 1, 2, 3, 4]
N_PER_CLASS = 200
SHOTS = 1000
N_CONST = 10
F_GRID = [7.27, 20.0, 50.0, 100.0]
RESULTS = Path("results")

raw = load_jets(features=("jetPt",), extra=BOOKKEEPING + PF_KEYS)
jets = select_quark_gluon(raw, require_loose_id=True)
y = jets["label"]
train, test = event_level_split(jets, test_fraction=0.3, seed=SEED)
z, d_eta, d_phi, _ = leading_constituents(jets, n_max=N_CONST)
d_r = np.sqrt(d_eta**2 + d_phi**2)
test_idx = np.flatnonzero(test)

## 1. Classical models on the 1P1Q inputs

In [ ]:
INPUTS = {
    "z, d_eta, d_phi (1P1Q inputs)": np.concatenate([z, d_eta, d_phi], axis=1),
    "z * d_eta, z * d_phi (1P1Q angles)": np.concatenate([z * d_eta, z * d_phi], axis=1),
    "z * dR (invariant angles)": z * d_r,
}
classical = {}
for name, F in INPUTS.items():
    res = {"rbf_svm": [], "bdt": []}
    for s in SEEDS:
        idx = balanced_subsample(y, train, N_PER_CLASS, seed=s)
        sc = StandardScaler().fit(F[idx])
        A, T = sc.transform(F[idx]), sc.transform(F[test_idx])
        res["rbf_svm"].append(roc_auc_score(y[test_idx], SVC(kernel="rbf", gamma="scale").fit(A, y[idx]).decision_function(T)))
        res["bdt"].append(roc_auc_score(y[test_idx], HistGradientBoostingClassifier(random_state=s).fit(A, y[idx]).predict_proba(T)[:, 1]))
    classical[name] = res
    print(f"{name:36s} RBF-SVM {np.mean(res['rbf_svm']):.3f} ± {np.std(res['rbf_svm']):.3f}   BDT {np.mean(res['bdt']):.3f} ± {np.std(res['bdt']):.3f}")

## 2. Rotation-invariant 1P1Q variant (quantum kernel)

In [ ]:
records = []
for f in F_GRID:
    for s in SEEDS:
        idx = balanced_subsample(y, train, N_PER_CLASS, seed=s)
        A, T = f * z[idx] * d_r[idx], f * z[test_idx] * d_r[test_idx]
        K_tr, K_te = angle_kernel(A, A), angle_kernel(T, A)
        mean_k, var_k = concentration(K_tr)
        rec = {"f": f, "seed": s, "kernel_mean": float(mean_k), "kernel_var": float(var_k), "eff_rank": effective_rank(K_tr)}
        rec["auc_exact"] = roc_auc_score(y[test_idx], SVC(kernel="precomputed").fit(K_tr, y[idx]).decision_function(K_te))
        rng = np.random.default_rng(1000 + s)
        svc = SVC(kernel="precomputed").fit(shot_noise(K_tr, SHOTS, rng, symmetric=True), y[idx])
        rec["auc_shots"] = roc_auc_score(y[test_idx], svc.decision_function(shot_noise(K_te, SHOTS, rng)))
        records.append(rec)

print(f"{'f':>6s} {'median |theta|':>15s} {'K mean':>8s} {'K var':>10s} {'eff. rank':>9s} {'AUC exact':>14s} {'AUC 1000 shots':>15s}")
for f in F_GRID:
    r = [x for x in records if x["f"] == f]
    g = lambda k: (np.mean([x[k] for x in r]), np.std([x[k] for x in r]))
    th = np.median(f * z[:, :4] * d_r[:, :4])
    print(f"{f:6.2f} {th:15.3f} {g('kernel_mean')[0]:8.4f} {g('kernel_var')[0]:10.2e} {g('eff_rank')[0]:9.2f} "
          f"{g('auc_exact')[0]:8.3f} ± {g('auc_exact')[1]:.3f} {g('auc_shots')[0]:9.3f} ± {g('auc_shots')[1]:.3f}")

json.dump({"classical": classical, "invariant_1p1q": records}, open(RESULTS / "onep1q_checks.json", "w"), indent=2)
print("saved results/onep1q_checks.json")

## 3. How to read this

- If classical models on the 1P1Q inputs also stay near 0.66, the gap to 0.80 comes from the **inputs** (10 constituents carry less than the 3 QGL variables), not from the quantum encoding.
- If they reach ~0.80 while the 1P1Q kernel stays at 0.66, the **encoding/kernel** loses information.
- If the rotation-invariant variant beats the original 1P1Q kernel, the orientation of constituents was acting as noise for a fixed kernel.